# Train a linear CLIP in NumPy

A toy world of colored shapes. Each photo is a feature vector built from how its color,
shape and size look, plus lighting that changes from photo to photo and some noise. Each
caption is a bag of words such as "a small red circle", and some captions leave out the
size. No training photo is a blue star, and no training caption says "blue star".

You will train two linear encoders and the temperature with the symmetric CLIP loss,
writing the backward pass yourself, and then classify blue stars zero-shot, from a prompt
the model has never seen.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

colors = ["red", "green", "blue", "yellow"]
shapes = ["circle", "square", "triangle", "star"]
sizes = ["small", "large"]
vocab = ["a", "photo", "of"] + sizes + colors + shapes
word_index = {w: i for i, w in enumerate(vocab)}

rng = np.random.default_rng(2026)
d_photo = 30
look = {w: rng.normal(0, 1, d_photo) for w in colors + shapes + sizes}   # how each attribute looks
light = rng.normal(0, 1, d_photo)                                        # lighting, different in every photo


def bag(words):
    """A caption as word counts over the vocabulary."""
    v = np.zeros(len(vocab))
    for w in words:
        v[word_index[w]] += 1
    return v


def make_pairs(n, seed, blue_stars=False):
    """n photo-caption pairs. Only blue stars when blue_stars is True, never otherwise."""
    r = np.random.default_rng(seed)
    X, C, kinds = [], [], []
    while len(X) < n:
        color, shape, size = colors[r.integers(4)], shapes[r.integers(4)], sizes[r.integers(2)]
        if ((color, shape) == ("blue", "star")) != blue_stars:
            continue
        x = look[color] + look[shape] + 0.5 * look[size] + r.normal(0, 3) * light + r.normal(0, 0.6, d_photo)
        words = ["a"] + ([size] if r.random() < 0.5 else []) + [color, shape]
        X.append(x)
        C.append(bag(words))
        kinds.append((color, shape))
    return np.array(X), np.array(C), kinds


X_train, C_train, kinds_train = make_pairs(3000, 1)
X_test, C_test, kinds_test = make_pairs(500, 2)
X_blue_stars, _, _ = make_pairs(100, 3, blue_stars=True)
print(f"photos {X_train.shape}, captions {C_train.shape}, held-out blue stars {X_blue_stars.shape}")

## Part 1 (coding task): Backpropagate through the scaling to length 1

Both encoders end by scaling each row to length 1: a row $x$ becomes $u = x / \lVert x \rVert$.
Write `normalize_backward(x, g)`. Its arguments are the rows $x$ before scaling, shape
$(n, k)$, and $g = \partial \mathcal{L} / \partial u$ for each row, the same shape. It
returns $\partial \mathcal{L} / \partial x$, row by row.

In [ ]:
def normalize_backward(x, g):
    """Gradient with respect to x of a loss whose gradient with respect to x / |x| is g."""
    x = np.asarray(x, dtype=float)
    g = np.asarray(g, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
grad_x = normalize_backward([[3.0, 4.0]], [[1.0, 0.0]])
assert np.shape(grad_x) == (1, 2), f"the result should have the shape of x, (1, 2), not {np.shape(grad_x)}"
assert np.allclose(grad_x, [[0.128, -0.096]]), f"x = (3, 4), g = (1, 0) gives {grad_x}, expected [[0.128, -0.096]]"
assert np.allclose(normalize_backward([[3.0, 4.0]], [[0.6, 0.8]]), 0), "a gradient along u itself must give 0: scaling x does not change x / |x|"
rng_check = np.random.default_rng(4)
x, g = rng_check.normal(size=(5, 4)) * [[1.0], [2.0], [0.5], [3.0], [1.0]], rng_check.normal(size=(5, 4))
D = rng_check.normal(size=(5, 4))
h = 1e-6
scaled = lambda rows: rows / np.linalg.norm(rows, axis=1, keepdims=True)
change = (np.sum(g * scaled(x + h * D)) - np.sum(g * scaled(x - h * D))) / (2 * h)
assert np.isclose(np.sum(normalize_backward(x, g) * D), change, rtol=1e-6), "on random rows of different lengths, the result does not match how the loss changes when x moves"
print("Part 1 passed")

## Part 2 (coding task): The loss and its gradients

The image encoder is $u_i = \text{scaled to length 1}(x_i W_i)$ and the text encoder is
$v_j = \text{scaled to length 1}(c_j W_t)$, where $c_j$ holds caption $j$'s word counts, so
$c_j W_t$ adds up the rows of $W_t$ for its words. The logits are
$Z = s\, U V^\top$ with the scale $s = \min(e^{\ell}, 100)$, where $\ell$ is the learned
parameter `log_scale` and the cap keeps the temperature $1/s$ at 0.01 or more. The loss is
the symmetric CLIP loss with the diagonal as the target.

Write `loss_and_grads(X, C, Wi, Wt, log_scale)`, returning `(loss, dWi, dWt, dlog_scale)`:
the loss, and its gradients with respect to `Wi`, `Wt` and `log_scale`. For one
cross-entropy, the gradient with respect to its logits is the softmax minus the target.

In [ ]:
def loss_and_grads(X, C, Wi, Wt, log_scale):
    """Symmetric CLIP loss of a batch, and its gradients with respect to Wi, Wt and log_scale."""
    # TODO
    raise NotImplementedError

In [ ]:
rng_check = np.random.default_rng(8)
Xc, Cc = rng_check.normal(size=(6, 5)), rng_check.integers(1, 3, (6, 4)).astype(float)
Wic, Wtc = rng_check.normal(size=(5, 3)), rng_check.normal(size=(4, 3))
loss, dWi, dWt, dls = loss_and_grads(Xc, Cc, Wic, Wtc, np.log(2.0))
assert np.isclose(loss, 2.399861, atol=1e-6), f"loss {loss:.6f} at scale 2, expected 2.399861: the average of the image-to-text and text-to-image cross-entropies"
assert np.shape(dWi) == Wic.shape and np.shape(dWt) == Wtc.shape, "dWi and dWt should have the shapes of Wi and Wt"


def rate(ls, Di, Dt, Dl, h=1e-6):
    """How fast the loss changes when the parameters move along (Di, Dt, Dl)."""
    up = loss_and_grads(Xc, Cc, Wic + h * Di, Wtc + h * Dt, ls + h * Dl)[0]
    down = loss_and_grads(Xc, Cc, Wic - h * Di, Wtc - h * Dt, ls - h * Dl)[0]
    return (up - down) / (2 * h)


for ls in (np.log(2.0), np.log(150.0)):
    loss, dWi, dWt, dls = loss_and_grads(Xc, Cc, Wic, Wtc, ls)
    Di, Dt = rng_check.normal(size=Wic.shape), rng_check.normal(size=Wtc.shape)
    for name, predicted, moved in [
        ("dWi", np.sum(dWi * Di), rate(ls, Di, 0 * Dt, 0.0)),
        ("dWt", np.sum(dWt * Dt), rate(ls, 0 * Di, Dt, 0.0)),
        ("dlog_scale", dls, rate(ls, 0 * Di, 0 * Dt, 1.0)),
    ]:
        assert np.isclose(predicted, moved, rtol=1e-4, atol=1e-7), f"with log_scale = ln {np.exp(ls):.0f}, {name} predicts the loss changes at rate {predicted:.6f}, but it changes at {moved:.6f}"
loss_capped = loss_and_grads(Xc, Cc, Wic, Wtc, np.log(150.0))[0]
assert np.isclose(loss_capped, 70.106592, atol=1e-6), f"loss {loss_capped:.6f} with log_scale = ln 150, expected 70.106592"
print("Part 2 passed")

## Part 3 (coding task): Train the encoders and the temperature

Write `train(X, C, steps=600, batch=128, lr=0.5, lr_scale=0.05, seed=0)`. Start from
`r = np.random.default_rng(seed)`, `Wi = r.normal(0, 0.1, (X.shape[1], 8))`,
`Wt = r.normal(0, 0.1, (C.shape[1], 8))` and a temperature of 0.5, that is
`log_scale = np.log(2.0)`. At every step, draw `batch` different rows with
`r.choice(len(X), batch, replace=False)`, compute the loss and the gradients on them, and
take a gradient descent step on all three parameters, with the learning rate `lr` for the
two matrices and `lr_scale` for `log_scale`. Return `(Wi, Wt, log_scale, losses)`, where
`losses` lists the loss of every step.

In [ ]:
def train(X, C, steps=600, batch=128, lr=0.5, lr_scale=0.05, seed=0):
    """Minibatch gradient descent on Wi, Wt and log_scale. Returns (Wi, Wt, log_scale, losses)."""
    # TODO
    raise NotImplementedError

In [ ]:
Wi, Wt, log_scale, losses = train(X_train, C_train)
losses = np.asarray(losses)
tau_learned = 1 / min(np.exp(log_scale), 100.0)
print(f"loss over the first 20 steps {losses[:20].mean():.3f}, over the last 50 {losses[-50:].mean():.3f}   (chance: ln 128 = {np.log(128):.3f})")
print(f"temperature: 0.5 at the start, {tau_learned:.3f} at the end")
assert len(losses) == 600, f"expected one loss per step, 600, got {len(losses)}"
assert losses[-50:].mean() < 2.3, f"the loss over the last 50 steps is {losses[-50:].mean():.3f}, expected below 2.3"
assert 0.03 < tau_learned < 0.2, f"the learned temperature is {tau_learned:.3f}: it should have moved well down from 0.5"
print("Part 3 passed")

## Part 4 (coding task): Blue stars, zero-shot

Write `zero_shot(X, prompts, Wi, Wt)`. `prompts` holds one bag of words per row. Return,
for every photo in `X`, the index of the prompt whose text embedding is most similar to the
photo's image embedding.

The check then classifies the 100 held-out blue stars among the 16 prompts "a {color}
{shape}", using the encoders you trained in Part 3.

In [ ]:
def zero_shot(X, prompts, Wi, Wt):
    """Index of the most similar prompt for every photo."""
    # TODO
    raise NotImplementedError

In [ ]:
pick = zero_shot([[1.0, 0.5]], [[1.0, 0.0], [0.0, 1.0]], np.eye(2), np.array([[1.0, 0.2], [3.0, 3.0]]))
assert np.shape(pick) == (1,), f"expected one index per photo, got shape {np.shape(pick)}"
assert pick[0] == 0, f"picked prompt {pick[0]}: prompt 0's embedding (1, 0.2) has the higher cosine with the photo, 0.965 against 0.949"

pairs = [(c, s) for c in colors for s in shapes]
prompts = np.array([bag(["a", c, s]) for c, s in pairs])
picked = [pairs[k] for k in zero_shot(X_blue_stars, prompts, Wi, Wt)]
acc_new = np.mean([p == ("blue", "star") for p in picked])
acc_test = np.mean([p == k for p, k in zip([pairs[k] for k in zero_shot(X_test, prompts, Wi, Wt)], kinds_test)])
r0 = np.random.default_rng(0)
acc_untrained = np.mean([pairs[k] == ("blue", "star") for k in zero_shot(X_blue_stars, prompts, r0.normal(0, 0.1, Wi.shape), r0.normal(0, 0.1, Wt.shape))])
print(f"blue stars labeled 'a blue star': {acc_new:.2f}   (untrained encoders: {acc_untrained:.2f}, guessing: {1 / 16:.2f})")
print(f"test photos, color and shape both right: {acc_test:.3f}")
assert acc_new >= 0.9 and acc_test >= 0.9, "expected at least 0.9 on both"
print("Part 4 passed")

## Solution

### Part 1: Backpropagate through the scaling to length 1

In [ ]:
def normalize_backward(x, g):
    """Gradient with respect to x of a loss whose gradient with respect to x / |x| is g."""
    x = np.asarray(x, dtype=float)
    g = np.asarray(g, dtype=float)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    u = x / norm
    radial = np.sum(g * u, axis=1, keepdims=True) * u     # the part of g along u
    return (g - radial) / norm

**Why it works.** Scaling $x$ by a positive number leaves $u$ unchanged, so the part of $g$ along $u$ can
have no effect on $x$ and is removed. What is left, the part of $g$ perpendicular to $u$,
moves $u$ by $1/\lVert x \rVert$ per unit that $x$ moves, hence the division. In matrix form
the Jacobian is $(I - uu^\top)/\lVert x \rVert$. Passing $g$ through unchanged, the
mistake that is easiest to make, is wrong both in size and in direction.

### Part 2: The loss and its gradients

In [ ]:
def loss_and_grads(X, C, Wi, Wt, log_scale):
    """Symmetric CLIP loss of a batch, and its gradients with respect to Wi, Wt and log_scale."""
    A, B = X @ Wi, C @ Wt
    U = A / np.linalg.norm(A, axis=1, keepdims=True)
    V = B / np.linalg.norm(B, axis=1, keepdims=True)
    capped = np.exp(log_scale) >= 100.0
    s = 100.0 if capped else np.exp(log_scale)
    cos = U @ V.T
    Z = s * cos
    n = len(Z)
    R = np.exp(Z - Z.max(1)[:, None])
    R /= R.sum(1)[:, None]                         # row softmaxes: each image over the captions
    K = np.exp(Z - Z.max(0)[None, :])
    K /= K.sum(0)[None, :]                         # column softmaxes: each caption over the images
    loss = -(np.mean(np.log(np.diag(R))) + np.mean(np.log(np.diag(K)))) / 2
    dZ = (R + K - 2 * np.eye(n)) / (2 * n)
    dcos = s * dZ
    dWi = X.T @ normalize_backward(A, dcos @ V)
    dWt = C.T @ normalize_backward(B, dcos.T @ U)
    dlog_scale = 0.0 if capped else float(np.sum(dZ * cos) * s)   # ds / d log_scale = s below the cap
    return loss, dWi, dWt, dlog_scale

**Why it works.** The chain runs backward one step at a time. The logits give $\partial\mathcal{L}/\partial Z
= (R + K - 2I)/(2N)$, with $R$ the row softmaxes and $K$ the column softmaxes, because each
logit sits in one row's cross-entropy and one column's. Since $Z = s\,UV^\top$, the
gradient with respect to $U$ is $s\,\frac{\partial\mathcal{L}}{\partial Z} V$ and with respect
to $V$ is $s\,\big(\frac{\partial\mathcal{L}}{\partial Z}\big)^{\top} U$. Part 1 carries these
through the scaling, and $A = XW_i$ gives $\partial\mathcal{L}/\partial W_i = X^\top\,
\partial\mathcal{L}/\partial A$. For the scale, $\partial\mathcal{L}/\partial s = \sum_{ij}
(\partial\mathcal{L}/\partial Z_{ij})\cos_{ij}$, and $ds/d\ell = e^{\ell} = s$ below the cap
and 0 above it, where the scale no longer moves with $\ell$. Learning $\ell = \ln s$
instead of $s$ keeps the scale positive and makes a step change it by a factor rather
than an amount.

### Part 3: Train the encoders and the temperature

In [ ]:
def train(X, C, steps=600, batch=128, lr=0.5, lr_scale=0.05, seed=0):
    """Minibatch gradient descent on Wi, Wt and log_scale. Returns (Wi, Wt, log_scale, losses)."""
    r = np.random.default_rng(seed)
    Wi = r.normal(0, 0.1, (X.shape[1], 8))
    Wt = r.normal(0, 0.1, (C.shape[1], 8))
    log_scale = np.log(2.0)
    losses = []
    for step in range(steps):
        rows = r.choice(len(X), batch, replace=False)
        loss, dWi, dWt, dls = loss_and_grads(X[rows], C[rows], Wi, Wt, log_scale)
        Wi = Wi - lr * dWi
        Wt = Wt - lr * dWt
        log_scale = log_scale - lr_scale * dls
        losses.append(loss)
    return Wi, Wt, log_scale, losses

**Why it works.** The loss cannot reach 0 here, for two reasons the lesson's
batch discussion predicts. Many captions leave out the size, so a batch of 128
often holds two pairs whose captions are the same bag of words, and the diagonal
target counts the other one as wrong. And at the starting temperature of 0.5
each logit is $2\cos$, so it lies between $-2$ and $2$. That bound alone is not
what floors the loss: with every off-diagonal cosine at $-1$ a batch of 128
would reach $\ln(1 + 127e^{-4}) \approx 1.2$. What keeps it near 2.9 is that the
128 photos fall on only 45 distinct caption bags, about 3 photos to a bag, so
each photo ties with about three others at the top logit, and that 45
directions in 8 dimensions cannot all point away from one another (their RMS
cosine is at least $\sqrt{(45-8)/(8 \cdot 44)} \approx 0.32$), so the off-diagonal
logits stay near $-0.6$ rather than near $-2$. The temperature falls as the encoders improve,
here to about 0.08, and that is much of the drop in the loss. Fixing it at 0.5
would stall the loss above 2.3 however good the encoders became.

### Part 4: Blue stars, zero-shot

In [ ]:
def zero_shot(X, prompts, Wi, Wt):
    """Index of the most similar prompt for every photo."""
    T = np.asarray(prompts, dtype=float) @ Wt
    T = T / np.linalg.norm(T, axis=1, keepdims=True)   # every prompt on the same footing
    return np.argmax(np.asarray(X, dtype=float) @ Wi @ T.T, axis=1)

**Why it works.** Scaling each prompt's embedding to length 1 matters: without it a prompt whose words
happen to have long vectors wins photos it does not match, as the check's two-prompt case
shows. Scaling the photo's embedding is optional for the choice, since it multiplies all
of one photo's scores by the same number. The model gets blue stars right although it never
saw one, because both encoders are linear: the text embedding of "a blue star" is the sum
of the word vectors for "a", "blue" and "star", each learned from other captions, and a
blue star's photo is the sum of how blue and how a star look. Real encoders are not linear,
but the same idea, labels built from parts the model has seen, is what lets CLIP classify
with label sets it was never trained on.